# Sentiment Analysis - YouTube Comments (Indonesian)
**Nama**: Asep Haryana Saputra

## 1. Import Libraries

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout, SpatialDropout1D, BatchNormalization
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils import resample
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import make_pipeline
from typing import Optional, Tuple, List, Dict
from tqdm.auto import tqdm

print(f"TensorFlow Version: {tf.__version__}")


2026-02-11 01:04:08.962125: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-11 01:04:08.993979: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-11 01:04:09.770873: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow Version: 2.20.0


## 2. Data Loading & Preprocessing (Advanced)
**Target Classes**:
- 0: **Negative**
- 1: **Positive**
- 2: **Spam**

In [2]:
"""
Data Preprocessing and Auto-Labeling Module for YouTube Comments Analysis
Optimized for specific dataset structure
Uses Transformers Library for High-Accuracy Labeling
"""

import pandas as pd
import numpy as np
import re
import os
from sklearn.utils import resample
from typing import Optional, Tuple, List, Dict
from transformers import pipeline # Transformers Import

class YouTubeCommentsPreprocessor:
    """Preprocessing and auto-labeling pipeline for YouTube comments dataset."""
    
    def __init__(self, filepath: str, random_seed: int = 42):
        """
        Initialize preprocessor with dataset filepath.
        """
        self.filepath = filepath
        self.random_seed = random_seed
        self.df = None
        self.labeled_data = None
        
        # Initialize Sentiment Library (Transformers)
        print("[INFO] Loading Transformers Sentiment Model (Indonesian RoBERTa)...")
        try:
            self.sentiment_model = pipeline(
                "sentiment-analysis", 
                model="w11wo/indonesian-roberta-base-sentiment-classifier",
                tokenizer="w11wo/indonesian-roberta-base-sentiment-classifier"
            )
            print("[INFO] Sentiment model loaded successfully")
        except Exception as e:
            print(f"[ERROR] Failed to load transformer model: {e}")
            self.sentiment_model = None
        
        # Minimal Spam Keywords (Remaining focused on explicit promo)
        self.spam_keywords = [
            'jual', 'obat', 'promo', 'sub4sub', 'giveaway', 'hadiah', 'klik',
            'top up', 'wa ', '08', 'judol', 'slot', 'donasi', 'saweria', 'wa:',
            'order', 'murah', 'diskon', 'cek ig', 'cek link', 'bio'
        ]
        
        # Special patterns for spam detection (Explicit URLs and Phones)
        self.spam_patterns = [
            r'https?://\S+',  # URLs
            r'\d{10,}',       # Phone numbers
        ]
        
        # Label mapping
        self.label_map = {
            0: 'negative',
            1: 'positive',
            2: 'spam',
            -1: 'unlabeled'
        }
    
    def load_dataset(self) -> pd.DataFrame:
        if not os.path.exists(self.filepath):
            raise FileNotFoundError(f"Dataset file not found at: {self.filepath}")
        self.df = pd.read_csv(self.filepath)
        print(f"[INFO] Loaded {len(self.df):,} rows from {self.filepath}")
        return self.df
    
    def filter_columns(self) -> pd.DataFrame:
        keep_columns = ['text', 'votes', 'heart']
        missing_cols = [col for col in keep_columns if col not in self.df.columns]
        if missing_cols:
            raise ValueError(f"Missing required columns: {missing_cols}")
        if 'label' in self.df.columns:
            # We will overwrite existing labels to ensure library quality
            print("[INFO] Overwriting existing labels with library predictions")
        self.df = self.df[keep_columns].copy()
        return self.df
    
    @staticmethod
    def clean_vote_count(vote_str: str) -> float:
        if pd.isna(vote_str) or vote_str == '': return 0.0
        vote_str = str(vote_str).strip().lower()
        multiplier = 1
        if 'rb' in vote_str: multiplier = 1000
        elif 'k' in vote_str: multiplier = 1000
        elif 'm' in vote_str: multiplier = 1_000_000
        vote_str = vote_str.replace('rb', '').replace('k', '').replace('m', '')
        vote_str = vote_str.replace(',', '').replace(' ', '')
        numeric_match = re.search(r'[\d.]+', vote_str)
        if numeric_match:
            try: return float(numeric_match.group()) * multiplier
            except: return 0.0
        return 0.0
    
    def standardize_heart_reaction(self) -> pd.DataFrame:
        heart_mapping = {'true': 1, 'false': 0, '1': 1, '0': 0, 'yes': 1, 'no': 0}
        self.df['heart'] = (
            self.df['heart'].astype(str).str.strip().str.lower()
            .map(heart_mapping).fillna(0).astype(int)
        )
        return self.df
    
    def _contains_spam_pattern(self, text: str) -> bool:
        text_lower = str(text).lower()
        for pattern in self.spam_patterns:
            if re.search(pattern, text_lower, re.IGNORECASE): return True
        return False
    
    def auto_label_comments(self, use_existing: bool = False) -> pd.DataFrame:
        """
        Label comments using Transformers library + Spam heuristics.
        """
        print("[INFO] Applying library-based auto-labeling (Transformers)...")
        print("[WARNING] This will process comments in batches. This may take 5-15 minutes.")
        
        if self.sentiment_model is None:
            raise ValueError("Model not loaded. Ensure transformers library is installed.")

        # Prepare texts
        texts = self.df['text'].fillna("").astype(str).tolist()
        processed_labels = []
        
        # Binary mapping for transformers output
        # Model w11wo outputs: 'positive', 'neutral', 'negative'
        trans_map = {
            'positive': 1,
            'negative': 0,
            'neutral': -1 # Treat as unlabeled unless spam caught it
        }

        # Process in batches for efficiency
        batch_size = 32
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            
            # Get predictions for batch
            try:
                preds = self.sentiment_model(batch, truncation=True, max_length=512)
            except Exception as e:
                print(f"Batch {i} failed: {e}")
                preds = [{'label': 'neutral'}] * len(batch)
            
            for j, p in enumerate(preds):
                comment = batch[j]
                comment_lower = comment.lower()
                
                # Check for SPAM first (Heuristics)
                is_spam = False
                if self._contains_spam_pattern(comment_lower):
                    is_spam = True
                elif any(k in comment_lower for k in self.spam_keywords):
                    is_spam = True
                elif len(comment_lower.strip()) < 3:
                    is_spam = True
                
                if is_spam:
                    processed_labels.append(2) # Spam
                else:
                    # Use Transformer prediction
                    sent_label = p['label']
                    processed_labels.append(trans_map.get(sent_label, -1))
            
            if (i // batch_size) % 10 == 0:
                print(f"Processed {i + len(batch)} / {len(texts)} comments...")

        self.df['label'] = processed_labels
        
        # Stats
        label_counts = self.df['label'].value_counts()
        print(f"[INFO] Auto-labeling completed:")
        for l, count in label_counts.items():
            name = self.label_map.get(l, "Unknown")
            print(f"  - {name}: {count}")
        
        return self.df
    
    def remove_unlabeled_comments(self) -> pd.DataFrame:
        if 'label' not in self.df.columns: raise ValueError("No label column")
        self.labeled_data = self.df[self.df['label'] != -1].copy()
        print(f"[INFO] Retained {len(self.labeled_data):,} labeled comments (excluding Neutral/Unlabeled)")
        return self.labeled_data
    
    def balance_dataset(self, strategy: str = 'upsample', min_samples: int = None) -> pd.DataFrame:
        if self.labeled_data is None: raise ValueError("No labeled data")
        
        df_pos = self.labeled_data[self.labeled_data['label'] == 1]
        df_neg = self.labeled_data[self.labeled_data['label'] == 0]
        df_spam = self.labeled_data[self.labeled_data['label'] == 2]
        
        print(f"distribution before balancing: Pos={len(df_pos)}, Neg={len(df_neg)}, Spam={len(df_spam)}")
        
        target = min_samples if min_samples else max(len(df_pos), len(df_neg), len(df_spam))
        
        if strategy == 'upsample':
            df_pos = resample(df_pos, replace=True, n_samples=target, random_state=42)
            df_neg = resample(df_neg, replace=True, n_samples=target, random_state=42)
            df_spam = resample(df_spam, replace=True, n_samples=target, random_state=42)
            self.labeled_data = pd.concat([df_pos, df_neg, df_spam])
            
        self.labeled_data = self.labeled_data.sample(frac=1, random_state=42).reset_index(drop=True)
        return self.labeled_data
    
    def clean_text_column(self) -> pd.DataFrame:
        def clean_text(text: str) -> str:
            if pd.isna(text): return ""
            text = str(text)
            text = re.sub(r'https?://\S+|www\.\S+', '', text)
            text = re.sub(r'@\w+', '', text)
            text = ' '.join(text.split())
            text = re.sub(r'[^\w\s\u00C0-\u017F]', ' ', text)
            return text.strip()
        
        target_df = self.labeled_data if self.labeled_data is not None else self.df
        target_df['text_clean'] = target_df['text'].apply(clean_text)
        return target_df

    def get_dataset_summary(self) -> Dict:
        if self.labeled_data is None: return {}
        return {
            'total': len(self.labeled_data),
            'distribution': self.labeled_data['label'].value_counts().to_dict()
        }
    
    def export_processed_data(self, output_path: str = None) -> str:
        if self.labeled_data is None: raise ValueError("No data")
        if not output_path: output_path = "processed_data_transformers.csv"
        self.labeled_data.to_csv(output_path, index=False)
        return output_path

    def run_pipeline(self, balance_strategy='upsample', min_samples=None, export=True):
        print("Running Transformers-based Pipeline...")
        self.load_dataset()
        self.filter_columns()
        self.df['votes'] = self.df['votes'].apply(self.clean_vote_count)
        self.standardize_heart_reaction()
        self.auto_label_comments()
        self.remove_unlabeled_comments()
        self.clean_text_column()
        self.balance_dataset(balance_strategy, min_samples)
        if export: self.export_processed_data()
        return self.labeled_data, self.get_dataset_summary() 


In [3]:

# Execute Preprocessing Pipeline
preprocessor = YouTubeCommentsPreprocessor(
    filepath='dataset_youtube.csv',
    random_seed=42
)

# Run pipeline (Upsample balancing enabled, export disabled for notebook flow)
df_clean, summary = preprocessor.run_pipeline(
    balance_strategy='upsample',
    export=False
)

print("\nSample Data:")
display(df_clean.head())


[INFO] Loading Transformers Sentiment Model (Indonesian RoBERTa)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: w11wo/indonesian-roberta-base-sentiment-classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Sentiment model loaded successfully
Running Transformers-based Pipeline...
[INFO] Loaded 15,500 rows from dataset_youtube.csv
[INFO] Applying library-based auto-labeling (Transformers)...
[WARNING] This will process comments in batches. This may take 5-15 minutes.
Processed 32 / 15500 comments...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processed 352 / 15500 comments...
Processed 672 / 15500 comments...
Processed 992 / 15500 comments...
Processed 1312 / 15500 comments...
Processed 1632 / 15500 comments...
Processed 1952 / 15500 comments...
Processed 2272 / 15500 comments...
Processed 2592 / 15500 comments...
Processed 2912 / 15500 comments...
Processed 3232 / 15500 comments...
Processed 3552 / 15500 comments...
Processed 3872 / 15500 comments...
Processed 4192 / 15500 comments...
Processed 4512 / 15500 comments...
Processed 4832 / 15500 comments...
Processed 5152 / 15500 comments...
Processed 5472 / 15500 comments...
Processed 5792 / 15500 comments...
Processed 6112 / 15500 comments...
Processed 6432 / 15500 comments...
Processed 6752 / 15500 comments...
Processed 7072 / 15500 comments...
Processed 7392 / 15500 comments...
Processed 7712 / 15500 comments...
Processed 8032 / 15500 comments...
Processed 8352 / 15500 comments...
Processed 8672 / 15500 comments...
Processed 8992 / 15500 comments...
Processed 9312 / 15500 

,text,votes,heart,label,text_clean
0,Wakakaka lucu kalii anjayyy,1.0,0,1,Wakakaka lucu kalii anjayyy
1,Challenge tiktok no bra viral..\nLangsung di c...,0.0,0,2,Challenge tiktok no bra viral Langsung di ce...
2,Up,0.0,0,2,Up
3,Up,0.0,0,2,Up
4,Anjayyy kok dilarang\nAnjayyy kok dipidana \nA...,34.0,0,2,Anjayyy kok dilarang Anjayyy kok dipidana Anja...


## 3. Modeling Schemes Comparison
We will test 3 different schemes ensuring variations in Algorithm, Feature Extraction, and Data Split.

1. **Scheme 1:** SVM + TF-IDF (80/20 Split)
2. **Scheme 2:** Random Forest + CountVectorizer (80/20 Split)
3. **Scheme 3:** Bi-LSTM + Embedding (70/30 Split) -> **Target > 92% Acc**

### Scheme 1: SVM + TF-IDF (80/20)

In [4]:

# Split Data 80/20
X_train_80, X_test_20, y_train_80, y_test_20 = train_test_split(
    df_clean['text_clean'], df_clean['label'], test_size=0.2, random_state=42, stratify=df_clean['label']
)

# Pipeline: TF-IDF -> SVM
pipeline_svm = make_pipeline(TfidfVectorizer(), SVC(kernel='linear', C=1.0, probability=True))
pipeline_svm.fit(X_train_80, y_train_80)

# Verify Accuracy
y_pred_svm = pipeline_svm.predict(X_test_20)
acc_svm = accuracy_score(y_test_20, y_pred_svm)
print(f"Scheme 1 (SVM) Accuracy: {acc_svm:.4f}")

unique_labels = sorted(list(set(y_test_20) | set(y_pred_svm)))
target_names = [name for i, name in enumerate(['Negative', 'Positive', 'Spam']) if i in unique_labels]
print(classification_report(y_test_20, y_pred_svm, labels=unique_labels, target_names=target_names))


Scheme 1 (SVM) Accuracy: 0.9116
              precision    recall  f1-score   support

    Negative       0.92      0.83      0.87      1829
    Positive       0.89      0.91      0.90      1830
        Spam       0.92      0.99      0.96      1830

    accuracy                           0.91      5489
   macro avg       0.91      0.91      0.91      5489
weighted avg       0.91      0.91      0.91      5489



### Scheme 2: Random Forest + CountVectorizer (80/20)

In [5]:

# Pipeline: CountVectorizer -> Random Forest
# Using the same 80/20 split as Scheme 1 but different Features and Algo
pipeline_rf = make_pipeline(CountVectorizer(), RandomForestClassifier(n_estimators=100, random_state=42))
pipeline_rf.fit(X_train_80, y_train_80)

# Verify Accuracy
y_pred_rf = pipeline_rf.predict(X_test_20)
acc_rf = accuracy_score(y_test_20, y_pred_rf)
print(f"Scheme 2 (Random Forest) Accuracy: {acc_rf:.4f}")

unique_labels = sorted(list(set(y_test_20) | set(y_pred_rf)))
target_names = [name for i, name in enumerate(['Negative', 'Positive', 'Spam']) if i in unique_labels]
print(classification_report(y_test_20, y_pred_rf, labels=unique_labels, target_names=target_names))


Scheme 2 (Random Forest) Accuracy: 0.9515
              precision    recall  f1-score   support

    Negative       0.96      0.90      0.93      1829
    Positive       0.97      0.95      0.96      1830
        Spam       0.93      1.00      0.96      1830

    accuracy                           0.95      5489
   macro avg       0.95      0.95      0.95      5489
weighted avg       0.95      0.95      0.95      5489



### Scheme 3: Deep Learning (Bi-LSTM) (70/30 Split)

In [6]:

VOCAB_SIZE = 5000
MAX_LENGTH = 100
EMBEDDING_DIM = 64

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df_clean['text_clean'])

sequences = tokenizer.texts_to_sequences(df_clean['text_clean'])
padded_sequences = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')

X_dl = padded_sequences
y_dl = df_clean['label'].values

# Split Data 70/30 (DIFFERENT SPLIT from Schemes 1 & 2)
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(X_dl, y_dl, test_size=0.3, random_state=42, stratify=y_dl)
print(f"DL Train shape: {X_train_dl.shape}, Test shape: {X_test_dl.shape}")


DL Train shape: (19208, 100), Test shape: (8233, 100)


In [7]:

model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.4),
    Bidirectional(LSTM(64)),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax') # 3 Output Classes
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Callbacks
checkpoint = ModelCheckpoint("best_model.keras", monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-6)

history = model.fit(
    X_train_dl, y_train_dl,
    epochs=20,
    batch_size=32,
    validation_data=(X_test_dl, y_test_dl),
    callbacks=[checkpoint, early_stop, reduce_lr]
)


Epoch 1/20


/home/asephs/venv/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1770746859.452811  980080 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2436 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-02-11 01:07:42.651985: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


600/601 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6243 - loss: 0.7511
Epoch 1: val_accuracy improved from None to 0.88522, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
601/601 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.7623 - loss: 0.5415 - val_accuracy: 0.8852 - val_loss: 0.2931 - learning_rate: 0.0010
Epoch 2/20
600/601 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9026 - loss: 0.2724
Epoch 2: val_accuracy improved from 0.88522 to 0.91923, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
601/601 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9081 - loss: 0.2597 - val_accuracy: 0.9192 - val_loss: 0.2259 - learning_rate: 0.0010
Epoch 3/20
600/601 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9272 - loss: 0.2111
Epoch 3: val_accuracy improved from 0.91923 to 0.92457, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
601/601 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - 

In [8]:

# Load best model
best_model = load_model("best_model.keras")
loss, acc_dl = best_model.evaluate(X_test_dl, y_test_dl)
print(f"Scheme 3 (Bi-LSTM) Accuracy: {acc_dl:.4f}")

y_pred_probs = best_model.predict(X_test_dl)
y_pred_dl = np.argmax(y_pred_probs, axis=1)

unique_labels = sorted(list(set(y_test_dl) | set(y_pred_dl)))
target_names = [name for i, name in enumerate(['Negative', 'Positive', 'Spam']) if i in unique_labels]
print(classification_report(y_test_dl, y_pred_dl, labels=unique_labels, target_names=target_names))


258/258 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9489 - loss: 0.1947
Scheme 3 (Bi-LSTM) Accuracy: 0.9489
258/258 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step
              precision    recall  f1-score   support

    Negative       0.93      0.93      0.93      2744
    Positive       0.94      0.92      0.93      2744
        Spam       0.97      1.00      0.99      2745

    accuracy                           0.95      8233
   macro avg       0.95      0.95      0.95      8233
weighted avg       0.95      0.95      0.95      8233



## 4. Comparison of Results

In [9]:

results = pd.DataFrame({
    'Model': ['SVM + TF-IDF', 'RF + BOW', 'Bi-LSTM'],
    'Split': ['80/20', '80/20', '70/30'],
    'Accuracy': [acc_svm, acc_rf, acc_dl],
    'Target > 85%': [acc_svm > 0.85, acc_rf > 0.85, acc_dl > 0.85],
    'DeepTarget > 92%': [False, False, acc_dl > 0.92]
})
print(results)


          Model  Split  Accuracy  Target > 85%  DeepTarget > 92%
0  SVM + TF-IDF  80/20  0.911641          True             False
1      RF + BOW  80/20  0.951539          True             False
2       Bi-LSTM  70/30  0.948864          True              True


## 5. Inference / Testing
Test the model with custom sentences.

In [10]:

# Re-implement basic cleaning for inference to match the pipeline
def clean_text_inference(text):
    text = str(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = ' '.join(text.split())
    text = re.sub(r'[^\w\s\u00C0-\u017F]', ' ', text)
    return text.strip()

def predict_sentiment(text_list):
    # Preprocess
    cleaned_texts = [clean_text_inference(t) for t in text_list]
    sequences = tokenizer.texts_to_sequences(cleaned_texts)
    padded = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')
    
    # Predict
    probs = best_model.predict(padded)
    preds = np.argmax(probs, axis=1)
    
    labels = {0: 'Negative', 1: 'Positive', 2: 'Spam'}
    results = []
    for t, p in zip(text_list, preds):
        results.append({'Text': t, 'Prediction': labels[p]})
    
    return pd.DataFrame(results)

# Test Cases
test_sentences = [
    "Video ini sangat bermanfaat, terima kasih bang!",
    "Konten sampah, tidak mendidik sama sekali.",
    "Cek ig kami kak, ada promo menarik.",
    "Biasa saja sih, tidak ada yang spesial.",
    "Gajelas lu bang, unsub ah.",
    "Haha,keren"
]

inference_results = predict_sentiment(test_sentences)
print(inference_results)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
                                              Text Prediction
0  Video ini sangat bermanfaat, terima kasih bang!   Positive
1       Konten sampah, tidak mendidik sama sekali.   Negative
2              Cek ig kami kak, ada promo menarik.       Spam
3          Biasa saja sih, tidak ada yang spesial.   Negative
4                       Gajelas lu bang, unsub ah.   Negative
5                                       Haha,keren   Positive
